Audio Embedding with Qdrant

In [22]:
import torch
import torchaudio
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct, VectorParams, Distance
import numpy as np


In [26]:
# Load the audio file
audio_path = r"C:\Users\admin\Desktop\qdrant_database\.ipynb_checkpoints\original-song-239607.mp3"
waveform, sample_rate = torchaudio.load(audio_path)

# Convert waveform to mono and resample if needed
if waveform.shape[0] > 1:
    waveform = torch.mean(waveform, dim=0, keepdim=True)

# Resample to 16kHz (common for audio models)
resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)
waveform = resampler(waveform)


In [34]:
import torchaudio
import torch

# Define the path to your audio file (use raw strings for Windows paths)
audio_path = r"C:\Users\admin\Desktop\qdrant_database\.ipynb_checkpoints\original-song-239607.mp3"
# Load the audio file
waveform, sample_rate = torchaudio.load(audio_path)

# Display basic information
print(f"Waveform Shape: {waveform.shape}")
print(f"Sample Rate: {sample_rate}")

# Convert to mono if it's stereo
if waveform.shape[0] > 1:
    waveform = torch.mean(waveform, dim=0, keepdim=True)
    print("Converted to mono.")

# Resample if the sample rate is not 16kHz (common for models like Wav2Vec2)
target_sample_rate = 16000
if sample_rate != target_sample_rate:
    resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=target_sample_rate)
    waveform = resampler(waveform)
    print(f"Resampled to {target_sample_rate} Hz.")

# Normalize the waveform
waveform = (waveform - waveform.mean()) / waveform.std()

print("Preprocessing completed.")



Waveform Shape: torch.Size([2, 7622784])
Sample Rate: 48000
Converted to mono.
Resampled to 16000 Hz.
Preprocessing completed.


In [36]:
# Compute the Mel Spectrogram (acts as an embedding)
mel_spectrogram = torchaudio.transforms.MelSpectrogram()(waveform)
embedding = torch.mean(mel_spectrogram, dim=-1).squeeze().numpy()


In [38]:
# Connect to Qdrant
client = QdrantClient("localhost", port=6333)

# Create an audio collection if it doesn't exist
client.recreate_collection(
    collection_name="audio_collection",
    vectors_config=VectorParams(size=len(embedding), distance=Distance.COSINE),
)

# Upload the audio embedding
client.upsert(
    collection_name="audio_collection",
    points=[
        PointStruct(id=1, vector=embedding, payload={"file_name": "audio1.wav"})
    ],
)


C:\Users\admin\AppData\Local\Temp\ipykernel_7112\1837041594.py:5: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [40]:
# Check the total number of vectors in the audio collection
stats = client.get_collection("audio_collection")
print(f"Total vectors in audio collection: {stats.vectors_count}")


Total vectors in audio collection: None


In [42]:
search_result = client.search(
    collection_name="audio_collection",
    query_vector=embedding,
    limit=1
)

print(search_result)


C:\Users\admin\AppData\Local\Temp\ipykernel_7112\1998573846.py:1: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  search_result = client.search(


[ScoredPoint(id=1, version=0, score=0.99999994, payload={'file_name': 'audio1.wav'}, vector=None, shard_key=None, order_value=None)]
